In [1]:
import ast
import pandas as pd

In [ ]:
input_path = ""
output_path = ""

In [ ]:
ls = "strict"
model = "medRoBERTa"
data = "baseline"

prediction_df = pd.read_csv(input_path)

In [116]:
def find_errors(df, model, data, ls):
    df = df.copy()

    df["categories"] = df["categories"].apply(ast.literal_eval)
    df["predicted_categories"] = df["predicted_categories"].apply(ast.literal_eval)

    errors = []

    for _, row in df.iterrows():
        true = row["categories"]
        pred = row["predicted_categories"]

        if ("B140 Attention functions" in true and "B152 Emotional functions" in pred) or \
        ("B455 Exercise tolerance functions" in true and "D450 Walking" in pred) or \
        ("D240 Handling stress and other psychological demands" in true and "B152 Emotional functions" in pred):
            row["model"] = model
            row["data"] = data
            row["ls"] = ls
            errors.append(row)

    errors = pd.DataFrame(errors)

    return errors

In [117]:
error_df = find_errors(prediction_df, model, data, ls)

Pick random sample

In [126]:
confused_cats = pd.read_csv("confusion_categories.csv")

In [127]:
error_types = [
    ("B140 Attention functions", "B152 Emotional functions"),
    ("B455 Exercise tolerance functions", "D450 Walking"),
    ("D240 Handling stress and other psychological demands", "B152 Emotional functions")
]

In [129]:
samples = []

for true_cat, pred_cat in error_types:
    error = confused_cats[
        confused_cats["categories"].apply(lambda x: true_cat in x) &
        confused_cats["predicted_categories"].apply(lambda x: pred_cat in x)
        ] 

    for data in ["baseline", "alldata", "convdata"]:
        samples.append(error[(error["model"] == "medRoBERTa") & (error["data"] == data) & (error["ls"] == "strict")].sample(3))
        samples.append(error[(error["model"] == "medRoBERTa") & (error["data"] == data) & (error["ls"] == "lenient")].sample(3))

        samples.append(error[(error["model"] == "mBERT") & (error["data"] == data) & (error["ls"] == "strict")].sample(3))
        samples.append(error[(error["model"] == "mBERT") & (error["data"] == data) & (error["ls"] == "lenient")].sample(3))

    samples.append(error[(error["model"] == "LLM") & (error["data"] == "zeroshot")].sample(3))
    samples.append(error[(error["model"] == "LLM") & (error["data"] == "fewshot")].sample(3))

sample_df = pd.concat(samples)
        

In [ ]:
sample_df.to_csv(output_path, index=False)